# Setup check

Run every cell (**Run → Run All Cells**). It takes about a minute.
At the end you should see a slider and a green message. If anything says
**MISSING** or **NOT FOUND**, see [`resources/troubleshooting.md`](resources/troubleshooting.md)
or ask a mentor.

## 1. Python and packages

In [ ]:
import importlib
import platform
import sys

print("Python", sys.version.split()[0], "on", platform.system())
assert sys.version_info >= (3, 10), "Please use Python 3.10 or newer"

core = ["ipywidgets", "anywidget", "numpy", "pandas", "sklearn", "matplotlib",
        "plotly", "build", "hatchling", "pytest"]
stations = ["jscatter", "drawdata", "quak", "altair", "lonboard", "ipyaladin", "vizarr"]


def check(names):
    problems = 0
    for name in names:
        try:
            module = importlib.import_module(name)
            version = getattr(module, "__version__", "installed")
            print(f"  OK       {name:12s} {version}")
        except Exception as error:  # noqa: BLE001
            problems += 1
            print(f"  MISSING  {name:12s} ({type(error).__name__}: {error})")
    return problems


print("Core packages (needed for every module):")
core_problems = check(core)
print("\nStation packages (Module 3, you need only the one you pick):")
station_problems = check(stations)

import anywidget  # noqa: E402

major, minor = (int(x) for x in anywidget.__version__.split(".")[:2])
if (major, minor) < (0, 11):
    core_problems += 1
    print("\nanywidget is too old: run  pip install -U 'anywidget>=0.11'")
print("\nCore:", "all good" if core_problems == 0 else f"{core_problems} problem(s)")

## 2. Kernel and server environment

Widgets need two installations: the Python packages in the **kernel**
environment, and the front-end extensions in the **Jupyter server**
environment. When they are different environments (common with conda), widgets
show as plain text such as `IntSlider(value=5)`.

In [ ]:
import pathlib
import shutil

print("Kernel Python :", sys.executable)
print("jupyter on PATH:", shutil.which("jupyter"))

labext = pathlib.Path(sys.prefix) / "share" / "jupyter" / "labextensions"
for name in ["@jupyter-widgets/jupyterlab-manager", "anywidget"]:
    found = (labext / name).exists()
    print(f"  {'found    ' if found else 'NOT FOUND'}  {name}  (in this kernel's environment)")

print("\nIf both are found but the widgets below show as text, JupyterLab is running")
print("from a DIFFERENT environment. Fix: install anywidget there too, or start")
print("JupyterLab from this environment. See resources/troubleshooting.md.")

## 3. Do widgets render?

In [ ]:
import anywidget
import ipywidgets as widgets
import traitlets


class SetupCheck(anywidget.AnyWidget):
    _esm = """
    export default {
      render({ model, el }) {
        el.textContent = "✅ anywidget works: " + model.get("message");
        el.style.color = "var(--jp-success-color1, green)";
      }
    };
    """
    message = traitlets.Unicode("custom widgets render here").tag(sync=True)


display(widgets.IntSlider(value=5, description="ipywidgets"), SetupCheck())

**You should see** a slider and a green line of text.

- Both visible: you are ready.
- Text such as `IntSlider(value=5, ...)` or `SetupCheck(...)`: the front-end
  extension is missing in the server environment (see section 2).
- Slider visible but no green text: open the browser console (F12) and show the
  error to a mentor.

## 4. Which parts need internet?

The core exercises work offline. Some Module 3 stations load code or data from
the internet in the browser:

| Station | Internet needed? |
|---|---|
| A. jupyter-scatter, B. drawdata, C. quak | No |
| D. Altair `JupyterChart` | Yes (loads Vega from a CDN) |
| E. Lonboard | Base map only (works without it) |
| F. ipyaladin | Yes (sky images) |
| G. vizarr | Yes (loads the viewer) |